# 04 – Modeling: Baseline & ML

**Projekt:** WealthScope AI  
**Kontext:** QUA3CK / Big-Data / Machine-Learning / Streamlit-App  
**Datenbasis:** Kaggle Stock/ETF Dataset, lokal verarbeitet  
**Hinweis:** Diese Notebooks dienen der wissenschaftlichen Nachvollziehbarkeit. Sie ersetzen keine Finanzberatung.

## Ziel

Dieses Notebook erstellt ein erstes ML-Baseline-Modell zur Zielvariable `target_20d`.

Wichtig:

- Das Modell ist eine Demonstration.
- Keine Anlageberatung.
- Ziel ist methodische Nachvollziehbarkeit.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

PARQUET_PATH = DATA_DIR / "wealthscope_features.parquet"
CSV_PATH = DATA_DIR / "wealthscope_features.csv"

def load_features():
    if PARQUET_PATH.exists():
        df = pd.read_parquet(PARQUET_PATH)
        source = "REAL_PARQUET"
    elif CSV_PATH.exists():
        df = pd.read_csv(CSV_PATH)
        source = "REAL_CSV"
    else:
        raise FileNotFoundError("Kein Feature-Datensatz gefunden. Erwartet wealthscope_features.parquet oder .csv")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    return df, source

df, source = load_features()

print("Datenquelle:", source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [ ]:
target = "target_20d"

feature_cols = [
    "daily_return",
    "return_5d",
    "return_20d",
    "ma_20_distance",
    "ma_50_distance",
    "ma_200_distance",
    "volatility_20d",
    "drawdown",
]

available_features = [c for c in feature_cols if c in df.columns]

model_df = df[available_features + [target]].dropna(subset=[target]).copy()
model_df[target] = model_df[target].astype(int)

print("Features:", available_features)
print("Model shape:", model_df.shape)
print("Target distribution:")
display(model_df[target].value_counts(normalize=True).rename("share"))

In [ ]:
X = model_df[available_features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

baseline_pred = np.repeat(y_train.mode().iloc[0], len(y_test))

baseline_scores = {
    "model": "Majority Baseline",
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall": recall_score(y_test, baseline_pred, zero_division=0),
    "f1": f1_score(y_test, baseline_pred, zero_division=0),
}

baseline_scores

In [ ]:
log_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

rf_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=150, random_state=42, max_depth=8)),
    ]
)

models = {
    "Logistic Regression": log_model,
    "Random Forest": rf_model,
}

results = [baseline_scores]

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    })

pd.DataFrame(results).sort_values("f1", ascending=False)

In [ ]:
best_model = rf_model
best_model.fit(X_train, y_train)
pred = best_model.predict(X_test)

print(classification_report(y_test, pred, zero_division=0))

cm = confusion_matrix(y_test, pred)
cm

In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix – Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.show()

In [ ]:
rf = best_model.named_steps["model"]
importances = pd.DataFrame({
    "feature": available_features,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

display(importances)

plt.figure(figsize=(8, 4))
plt.bar(importances["feature"], importances["importance"])
plt.title("Feature Importance – Random Forest")
plt.xticks(rotation=45, ha="right")
plt.show()